# 17 · Phase 15 — Ablations of the full deep model

Each row changes **one** ingredient of the proposed system and retrains the tri-branch network
(seed 0, identical training recipe). Policy-changing ablations re-run the selected RL agent with
the restricted action space / modified reward (Notebook 14) and use its best-validation policy.

| ablation | what is removed |
|---|---|
| no RL | fixed baseline configuration for every recording (= B5) |
| fixed wavelet / band / window | that component pinned to its baseline value; RL chooses the rest |
| no wavelet branch | fusion restricted to T, S, TS |
| concatenation fusion | same RL policy, attention fusion replaced by concatenation + linear layer |
| accuracy-only reward | reward = balanced term only (no sens/spec/robustness/complexity terms) |
| no robustness term / no complexity penalty | that reward term set to 0 |
| no state adaptation (K = 1) | one global RL-selected configuration |
| classical reward oracle | policy learned from the logistic-regression oracle instead of the supernet |

In [ ]:
%run common/00_config.ipynb
%run common/01_signal_lib.ipynb
%run common/02_rl_lib.ipynb
%run common/03_deep_lib.ipynb
import torch

device = torch.device("cuda")
space = ConfigSpace(CFG)
meta, signals = load_deep_data(P)
quiet = lambda s: print(s) if s.startswith("[") else None
S0 = (0,)

rows = []
fpol = load_json(P["results"] / "policies" / "rl_full.json")
cfg_full = policy_configs(meta, space, np.array(fpol["policy"]), fpol["K"])
rows += run_experiment("R2_full_rl_frozen_policy", meta, signals, cfg_full, CFG, P, device, seeds=S0, log=quiet)
rows.append(dict(load_summary("B5_fixed_TSW_fusion", 0, P), model="A_no_rl (B5 fixed config)"))
rows += run_experiment("A_concat_fusion", meta, signals, cfg_full, CFG, P, device, seeds=S0, fusion="concat", log=quiet)
for slug in ["abl_fixed_wavelet", "abl_fixed_band", "abl_fixed_window", "abl_no_wavelet_branch", "abl_accuracy_reward",
             "abl_no_robustness", "abl_no_complexity", "abl_K1", "abl_classical_oracle"]:
    pol = load_json(P["results"] / "policies" / f"{slug}.json")
    cfgs = policy_configs(meta, space, np.array(pol["policy"]), pol["K"])
    rows += run_experiment("A_" + slug[4:], meta, signals, cfgs, CFG, P, device, seeds=S0, log=quiet)

res = pd.DataFrame(rows)
res.to_csv(P["tables"] / "17_deep_ablations_runs.csv", index=False)
cols = ["macro_f1", "sensitivity", "specificity", "balanced_acc", "auroc", "auprc"]
tab = res.set_index("model")[[f"test_{c}" for c in cols] + ["val_auroc"]]
tab.columns = [c.replace("test_", "") for c in tab.columns]
ref = tab.loc["R2_full_rl_frozen_policy"]
tab["Δ macro-F1 vs full"] = tab.macro_f1 - ref.macro_f1
tab["Δ AUROC vs full"] = tab.auroc - ref.auroc
tab.to_csv(P["tables"] / "17_deep_ablations.csv")
tab.round(3)